# Vector Databases & FAISS (ANN Indexing) — AI Lab Instructor Notebook

*RAG & Retrieval · Hard*



5 tasks, all with solutions.

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
import numpy as np
from collections import defaultdict

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)


## Task 1: Dataset & Setup

# Vector Databases & FAISS (ANN) — Student Lab

Offline ANN lab: brute force vs IVF (inverted file) indexing.

## Section 0 — Synthetic embeddings + queries

In [ ]:
def l2_normalize(X: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    nrm = np.sqrt(np.sum(X*X, axis=1, keepdims=True))
    return X / (nrm + eps)

def make_clustered(n: int = 20000, d: int = 64, n_clusters: int = 200, seed: int = 0):
    rng = np.random.default_rng(seed)
    centers = rng.standard_normal((n_clusters, d))
    centers = l2_normalize(centers)
    # choose cluster for each point
    cids = rng.integers(0, n_clusters, size=(n,))
    X = centers[cids] + 0.15 * rng.standard_normal((n, d))
    X = l2_normalize(X)
    return X, cids, centers

X, true_cids, true_centers = make_clustered(n=12000, d=64, n_clusters=128, seed=0)
print('X', X.shape)
check('norm1', float(np.max(np.abs(np.sqrt(np.sum(X*X, axis=1)) - 1.0))) < 1e-6)

# Queries: sample some vectors + small noise
nq = 200
qid = rng.choice(X.shape[0], size=(nq,), replace=False)
Q = X[qid] + 0.05 * rng.standard_normal((nq, X.shape[1]))
Q = l2_normalize(Q)


**Why this works.** # Vector Databases & FAISS (ANN) — Instructor Lab

Offline ANN lab: brute force vs IVF (inverted file) indexing.

## Section 0 — Synthetic embeddings + queries

## Task 2: Section 1 — Brute-force cosine top-k (ground truth)
*Section: Brute-force cosine top-k (ground truth)*

## Section 1 — Brute-force cosine top-k (ground truth)

In [ ]:
def cosine_sim(A: np.ndarray, B: np.ndarray) -> np.ndarray:
    # A:(m,d), B:(n,d) -> (m,n) dot product if already normalized
    return A @ B.T

def topk_rows(S: np.ndarray, k: int):
    # returns indices (m,k)
    return np.argsort(-S, axis=1)[:, :k]

k = 10
S = cosine_sim(Q, X)
gt = topk_rows(S, k=k)
print('gt[0][:5]', gt[0,:5])
check('gt_shape', gt.shape == (Q.shape[0], k))


In [ ]:
# Checks
check('gt_shape', gt.shape == (Q.shape[0], k))

## Task 3: Section 2 — K-means (coarse quantizer)
*Section: K-means (coarse quantizer)*

## Section 2 — K-means (coarse quantizer)

In [ ]:
def kmeans(X: np.ndarray, k: int, iters: int = 15, seed: int = 0):
    rng = np.random.default_rng(seed)
    n, d = X.shape
    cent = X[rng.choice(n, size=(k,), replace=False)].copy()
    for _ in range(iters):
        # assign
        sims = X @ cent.T  # cosine-like if normalized
        labels = np.argmax(sims, axis=1)
        # update
        for j in range(k):
            idx = np.where(labels == j)[0]
            if idx.size == 0:
                cent[j] = X[rng.integers(0, n)]
            else:
                c = np.mean(X[idx], axis=0)
                cent[j] = c / (np.sqrt(np.sum(c*c)) + 1e-12)
    return cent, labels

nlist = 128  # IVF lists
cent, labels = kmeans(X, k=nlist, iters=12, seed=0)
check('cent_shape', cent.shape == (nlist, X.shape[1]))


In [ ]:
# Checks
check('cent_shape', cent.shape == (nlist, X.shape[1]))

## Task 4: Section 3 — Build IVF index
*Section: Build IVF index*

## Section 3 — Build IVF index

In [ ]:
def build_ivf(labels: np.ndarray, nlist: int):
    lists = [[] for _ in range(nlist)]
    for i, c in enumerate(labels.tolist()):
        lists[c].append(i)
    return [np.array(lst, dtype=np.int64) for lst in lists]

ivf_lists = build_ivf(labels, nlist=nlist)
sizes = np.array([len(lst) for lst in ivf_lists])
print('avg_list_size', float(np.mean(sizes)), 'max', int(np.max(sizes)))
check('lists_ok', sum(sizes) == X.shape[0])


In [ ]:
# Checks
check('lists_ok', sum(sizes) == X.shape[0])

## Task 5: Section 4 — IVF query (n_probe)
*Section: IVF query (n_probe)*

## Section 4 — IVF query (n_probe)

In [ ]:
$f